In [4]:
# DRAW CANVAS

import cv2
import numpy as np
import base64
import time

from IPython.display import display, Javascript
from google.colab.output import eval_js


# =========================================================
# BLUE COLOR RANGE
# =========================================================

LOWER_COLOR = np.array([100, 150, 0])
UPPER_COLOR = np.array([140, 255, 255])


# =========================================================
# JAVASCRIPT CAMERA + AIR CANVAS
# =========================================================

display(Javascript("""
(async function() {

    // Remove previous canvas if present
    const old = document.getElementById("airCanvasApp");
    if (old) old.remove();

    const app = document.createElement("div");
    app.id = "airCanvasApp";
    app.style.textAlign = "center";

    const title = document.createElement("h2");
    title.innerText = "🎨 BCA AI - Virtual Air Canvas";
    app.appendChild(title);

    const video = document.createElement("video");
    video.width = 640;
    video.height = 480;
    video.autoplay = true;
    video.style.border = "2px solid black";

    app.appendChild(video);

    const canvas = document.createElement("canvas");
    canvas.width = 640;
    canvas.height = 480;
    canvas.style.position = "absolute";
    canvas.style.left = "0px";
    canvas.style.top = "0px";
    canvas.style.pointerEvents = "none";

    const wrapper = document.createElement("div");
    wrapper.style.position = "relative";
    wrapper.style.width = "640px";
    wrapper.style.height = "480px";
    wrapper.style.margin = "auto";

    wrapper.appendChild(video);
    wrapper.appendChild(canvas);

    // Replace video position
    app.innerHTML = "";
    app.appendChild(title);
    app.appendChild(wrapper);

    const buttons = document.createElement("div");
    buttons.style.marginTop = "15px";

    const clearButton = document.createElement("button");
    clearButton.innerText = "🧹 Clear Canvas";
    clearButton.style.padding = "10px";
    clearButton.style.margin = "5px";

    const stopButton = document.createElement("button");
    stopButton.innerText = "⛔ Stop Camera";
    stopButton.style.padding = "10px";
    stopButton.style.margin = "5px";

    buttons.appendChild(clearButton);
    buttons.appendChild(stopButton);
    app.appendChild(buttons);

    document.body.appendChild(app);

    try {

        const stream =
            await navigator.mediaDevices.getUserMedia({
                video: {
                    width: 640,
                    height: 480
                },
                audio: false
            });

        video.srcObject = stream;

        window.airCanvasStream = stream;
        window.airCanvasVideo = video;
        window.airCanvasCanvas = canvas;

        clearButton.onclick = function() {
            const ctx = canvas.getContext("2d");
            ctx.clearRect(0, 0, canvas.width, canvas.height);
        };

        stopButton.onclick = function() {
            stream.getTracks().forEach(track => track.stop());
            app.remove();

            window.airCanvasStream = null;
            window.airCanvasVideo = null;
        };

    } catch (err) {

        alert(
            "Camera permission denied! " +
            err.message
        );

    }

})();
"""))

print("Camera starting...")
time.sleep(2)

print("If browser asks for camera permission, click ALLOW.")


<IPython.core.display.Javascript object>

Camera starting...
If browser asks for camera permission, click ALLOW.


In [ ]:
# OBJECT DETECTION

In [5]:
!pip install ultralytics opencv-python

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.7/46.7 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 24.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.1/58.1 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.0/91.0 kB 8.8 MB/s eta 0:00:00


In [ ]:
import cv2
import numpy as np
import base64
import time

from ultralytics import YOLO
from IPython.display import display, Javascript
from google.colab.output import eval_js

# ============================================================
# YOLO MODEL
# ============================================================

model = YOLO("yolov8n.pt")

print("YOLO model loaded successfully!")


# ============================================================
# START BROWSER CAMERA
# ============================================================

display(Javascript("""
(async function() {

    // Remove old camera if present
    const old = document.getElementById("yoloCamera");
    if (old) {
        old.remove();
    }

    const container = document.createElement("div");
    container.id = "yoloCamera";
    container.style.textAlign = "center";

    const title = document.createElement("h2");
    title.innerText = "🤖 YOLO Object Detection";
    container.appendChild(title);

    const video = document.createElement("video");

    video.width = 640;
    video.height = 480;
    video.autoplay = true;
    video.style.border = "3px solid blue";

    container.appendChild(video);

    const stopButton = document.createElement("button");
    stopButton.innerText = "⛔ Stop Camera";
    stopButton.style.display = "block";
    stopButton.style.margin = "10px auto";
    stopButton.style.padding = "10px 20px";

    container.appendChild(stopButton);

    document.body.appendChild(container);

    try {

        const stream =
            await navigator.mediaDevices.getUserMedia({
                video: {
                    width: 640,
                    height: 480
                },
                audio: false
            });

        video.srcObject = stream;

        window.yoloStream = stream;
        window.yoloVideo = video;

        stopButton.onclick = function() {

            stream.getTracks().forEach(
                track => track.stop()
            );

            container.remove();

            window.yoloStream = null;
            window.yoloVideo = null;
        };

    } catch (error) {

        alert(
            "Camera permission denied: " +
            error.message
        );

    }

})();
"""))

time.sleep(3)

print("Camera started.")
print("If browser asks for permission, click ALLOW.")

import matplotlib.pyplot as plt
from IPython.display import clear_output

print("Starting YOLO detection...")
print("Detection will run for 30 seconds.")

start_time = time.time()

while time.time() - start_time < 30:

    # Get frame from browser camera
    data = eval_js("""
    (async function() {

        if (!window.yoloVideo) {
            return null;
        }

        const video = window.yoloVideo;

        const canvas = document.createElement("canvas");

        canvas.width = video.videoWidth;
        canvas.height = video.videoHeight;

        const ctx = canvas.getContext("2d");

        ctx.drawImage(
            video,
            0,
            0,
            canvas.width,
            canvas.height
        );

        return canvas.toDataURL(
            "image/jpeg",
            0.7
        );

    })()
    """)

    if data is None:
        print("Camera stopped.")
        break

    # Decode image
    image_bytes = base64.b64decode(
        data.split(",")[1]
    )

    image_array = np.frombuffer(
        image_bytes,
        dtype=np.uint8
    )

    frame = cv2.imdecode(
        image_array,
        cv2.IMREAD_COLOR
    )

    # ========================================================
    # YOLO DETECTION
    # ========================================================

    results = model(
        frame,
        verbose=False
    )

    result = results[0]

    # Draw detection boxes
    annotated_frame = result.plot()

    # Count objects
    detected_count = len(result.boxes)

    # Add total count
    cv2.putText(
        annotated_frame,
        f"Total Objects: {detected_count}",
        (20, 40),
        cv2.FONT_HERSHEY_SIMPLEX,
        1,
        (0, 255, 0),
        2
    )

    # Convert BGR -> RGB
    annotated_frame = cv2.cvtColor(
        annotated_frame,
        cv2.COLOR_BGR2RGB
    )

    # Display
    clear_output(wait=True)

    plt.figure(
        figsize=(10, 7)
    )

    plt.imshow(annotated_frame)
    plt.axis("off")
    plt.show()

print("Detection finished.")

